# Download de Documentos — Levantamento de Políticas de IA em Universidades Brasileiras

Este notebook lê o arquivo CSV do levantamento e baixa automaticamente cada documento disponível.

**Estrutura esperada do projeto:**
```
ai-governance-brazilian-universities/
├── data/
│   ├── raw/
│   │   └── levantamento_ia_universidades_brasileiras.csv
│   └── processed/
│       └── documentos/      ← todos os PDFs ficam aqui, sem subpastas
├── notebooks/
│   └── download_documentos_ia.ipynb
├── scripts/
└── outputs/
```

**Convenção de nomenclatura dos arquivos:**
`INST_ANO_TIPO.pdf` — ex: `UFG_2024_Guia.pdf`

O CSV `levantamento_ia_universidades_brasileiras.csv` é a fonte de verdade sobre cada documento:
instituição, região, categoria, ano, tipo e status. A pasta `documentos/` é apenas repositório físico.

**Pré-requisitos:**
```bash
pip install requests pandas tqdm
```

## 1. Imports e configurações gerais

Usamos `Path.cwd()` para localizar a raiz do projeto a partir do diretório de trabalho do Jupyter,
que ao rodar no VS Code corresponde à pasta `notebooks/`. O `.parent` sobe um nível para a raiz.

In [ ]:
import os
import re
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from urllib.parse import urlparse

# Raiz do projeto — sobe um nível a partir de notebooks/
ROOT = Path.cwd().parent

# Caminhos derivados da estrutura do projeto
CSV_PATH  = ROOT / "data" / "raw" / "levantamento_ia_universidades_brasileiras.csv"
DOCS_PATH = ROOT / "data" / "processed" / "documentos"
LOG_PATH  = ROOT / "data" / "processed" / "log_downloads.csv"

# Tempo máximo de espera por download (segundos)
TIMEOUT = 30

# Intervalo entre requisições — evita sobrecarregar os servidores
INTERVALO = 1.5

# Headers que simulam um navegador comum
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/pdf,text/html,application/xhtml+xml,*/*",
}

print(f"Raiz do projeto : {ROOT}")
print(f"CSV de entrada  : {CSV_PATH}")
print(f"Pasta de saída  : {DOCS_PATH}")
print(f"Log             : {LOG_PATH}")
print()
print("✅ Configurações carregadas.")

## 2. Leitura e pré-visualização do CSV

Carregamos o arquivo de `data/raw/` e fazemos um diagnóstico inicial:
quantos registros têm URL disponível e quais precisarão de busca manual.

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"Total de registros no CSV : {len(df)}")
print(f"Com URL disponível        : {df['url'].notna().sum()}")
print(f"Sem URL (busca manual)    : {df['url'].isna().sum()}")
print()

# Lista os registros sem URL para referência
sem_url = df[df['url'].isna() | (df['url'].str.strip() == '')]
if not sem_url.empty:
    print("📋 Registros sem URL — requerem busca manual:")
    display(sem_url[['instituicao', 'ano', 'tipo_documento', 'status']])

# Trabalhamos apenas com os que têm URL
df_com_url = df[df['url'].notna() & (df['url'].str.strip() != '')].copy()
df_com_url = df_com_url.reset_index(drop=True)

print(f"\n✅ {len(df_com_url)} documentos prontos para download.")

## 3. Criação da pasta de destino

Todos os documentos vão para `data/processed/documentos/` — sem subpastas por região.
O CSV é a fonte de verdade sobre a procedência de cada arquivo.

In [ ]:
DOCS_PATH.mkdir(parents=True, exist_ok=True)
print(f"📁 {DOCS_PATH.relative_to(ROOT)}")
print("\n✅ Pasta de destino pronta.")

## 4. Funções auxiliares

- **`inferir_extensao`** — identifica o tipo do arquivo pela URL ou pelo Content-Type da resposta HTTP.
- **`nome_arquivo`** — gera um nome padronizado no formato `INST_ANO_TIPO`, sem caracteres especiais.

In [ ]:
def inferir_extensao(url, response):
    """Determina a extensão do arquivo pela URL ou pelo Content-Type da resposta."""
    path = urlparse(url).path.lower()
    for ext in (".pdf", ".docx", ".doc"):
        if path.endswith(ext):
            return ext
    content_type = response.headers.get("Content-Type", "")
    if "pdf" in content_type:
        return ".pdf"
    if "word" in content_type or "officedocument" in content_type:
        return ".docx"
    if "html" in content_type:
        return ".html"
    return ".bin"


def nome_arquivo(row):
    """Gera nome de arquivo padronizado: INST_ANO_TIPO (sem caracteres especiais)."""
    inst = re.sub(r"[^\w]", "_", str(row['instituicao']))
    tipo = re.sub(r"[^\w]", "_", str(row['tipo_documento'])[:30])
    return f"{inst}_{row['ano']}_{tipo}"


print("✅ Funções auxiliares definidas.")

## 5. Download dos documentos

Loop principal. Para cada registro com URL:

1. Faz a requisição HTTP
2. Verifica o código de resposta
3. Infere a extensão do arquivo
4. Se for HTML (página web), registra como revisão manual — não salva
5. Caso contrário, salva o arquivo em `data/processed/documentos/`
6. Aguarda o intervalo antes da próxima requisição

Todos os resultados são acumulados no `log` para o relatório final.

In [ ]:
log = []

for _, row in tqdm(df_com_url.iterrows(), total=len(df_com_url), desc="Baixando documentos"):
    url = str(row['url']).strip()

    resultado = {
        "instituicao": row['instituicao'],
        "ano"        : row['ano'],
        "url"        : url,
        "status"     : None,
        "arquivo"    : None,
        "observacao" : ""
    }

    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True)

        if response.status_code != 200:
            resultado["status"]     = "❌ Erro HTTP"
            resultado["observacao"] = f"Status {response.status_code}"
            log.append(resultado)
            time.sleep(INTERVALO)
            continue

        ext = inferir_extensao(url, response)

        # URL aponta para página web — não há arquivo para baixar
        if ext == ".html":
            resultado["status"]     = "⚠️ Página HTML"
            resultado["observacao"] = "URL aponta para página web. Acesse manualmente para localizar o documento."
            log.append(resultado)
            time.sleep(INTERVALO)
            continue

        # Salva o arquivo em data/processed/documentos/
        nome_final = nome_arquivo(row) + ext
        caminho    = DOCS_PATH / nome_final

        with open(caminho, "wb") as f:
            f.write(response.content)

        resultado["status"]  = "✅ Baixado"
        resultado["arquivo"] = str(caminho.relative_to(ROOT))

    except requests.exceptions.Timeout:
        resultado["status"]     = "❌ Timeout"
        resultado["observacao"] = "Servidor não respondeu no tempo limite."
    except requests.exceptions.ConnectionError:
        resultado["status"]     = "❌ Erro de conexão"
        resultado["observacao"] = "Não foi possível conectar ao servidor."
    except Exception as e:
        resultado["status"]     = "❌ Erro inesperado"
        resultado["observacao"] = str(e)

    log.append(resultado)
    time.sleep(INTERVALO)

print("\n✅ Loop de downloads concluído.")

## 6. Relatório final

Resumo dos resultados. O log completo é salvo em `data/processed/log_downloads.csv`.
Documentos que precisam de download manual estão identificados na tabela abaixo.

In [ ]:
df_log = pd.DataFrame(log)

resumo = df_log['status'].value_counts()

print("=" * 52)
print(" RELATÓRIO FINAL")
print("=" * 52)
for status, count in resumo.items():
    print(f"  {status}: {count}")
print("=" * 52)

# Detalha os que precisam de atenção manual
problemas = df_log[df_log['status'] != "✅ Baixado"]
if not problemas.empty:
    print("\n📋 Requerem atenção manual:")
    display(problemas[['instituicao', 'ano', 'status', 'observacao', 'url']])

# Salva o log em data/processed/
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
df_log.to_csv(LOG_PATH, index=False, encoding='utf-8-sig')
print(f"\n📄 Log salvo em: {LOG_PATH.relative_to(ROOT)}")

## 7. Verificação final — arquivos em `data/processed/documentos/`

Lista todos os arquivos baixados com seus tamanhos em KB.

In [ ]:
print(f"📂 {DOCS_PATH.relative_to(ROOT)}\n")

arquivos       = sorted(DOCS_PATH.glob("*"))
total_bytes    = 0
total_arquivos = 0

for arq in arquivos:
    kb           = arq.stat().st_size / 1024
    total_bytes += arq.stat().st_size
    total_arquivos += 1
    print(f"  {arq.name}  ({kb:.1f} KB)")

print(f"\nTotal: {total_arquivos} arquivo(s) — {total_bytes/1024:.1f} KB")